# Battle Lab · Entrenamiento M-C — VGC-Bench sin piedad 🐉⚔️

Libreta canónica de entrenamiento para `BATTLE-LAB-MC-TRAIN-001`.

Esta fase conserva intacto el checkpoint público M-A/M-B de **VGC-Bench**, construye un corpus de **Champions M-C** desde **VGCPastes Repository**, intenta reunir replays OTS M-C con el mismo filtro que usa VGC-Bench para Behavior Cloning, audita descartes/Elo/cobertura/espacio y después, solo en modos de entrenamiento, continúa el aprendizaje con PPO/self-play sobre el corpus M-C.

**Fuente de equipos:** `VGCPastes Repository → Champions M-C` (184 equipos al 13-sep-2026, todos con Poképaste y EVs según el censo inicial). Cada snapshot vuelve a descargarse y pasa por el validador del Showdown fijado por el proyecto antes de entrar al entrenamiento.

**Importante:** el dataset oficial `cameronangliss/vgc-battle-logs` todavía contiene M-A/M-B, no M-C. Por eso esta libreta reconstruye la misma clase de datos desde Showdown cuando estén disponibles. Si todavía hay pocas trayectorias humanas M-C, no inventa demostraciones: salta BC-MC y arranca RL desde el checkpoint oficial M-A/M-B.

Todos los datos, manifests, checkpoints y resultados quedan persistentes en Drive bajo `Colabs/LikeNoOneEverWas/BattleLab/MC-Training/`. El runtime temporal de `/content` puede borrarse sin perder el trabajo.


## 1. Configuración

`CENSUS` es la primera ejecución recomendada y obligatoria antes del piloto: actualiza equipos/replays/trayectorias, audita descartes, Elo, cobertura y espacio, y no entrena. Cuando ese censo quede revisado, cambia a `LIGHT` para el primer piloto. `NORMAL` y `HARD` amplían el self-play. El actor de VGC-Bench permanece congelado durante los primeros 98,304 pasos cuando parte de BC, por eso LIGHT usa 196,608 pasos.


In [ ]:
RUN_MODE = "CENSUS"  # @param ["CENSUS", "LIGHT", "NORMAL", "HARD"]
SYNC_TEAMS = True  # @param {type:"boolean"}
SCRAPE_HUMAN_LOGS = True  # @param {type:"boolean"}
BUILD_TRAJECTORIES = True  # @param {type:"boolean"}
RUN_BC_IF_ENOUGH_DATA = True  # @param {type:"boolean"}
RUN_RL = True  # @param {type:"boolean"}
MIN_BC_TRAJECTORIES = 1000  # @param {type:"integer"}
MIN_BC_TRANSITIONS = 10000  # @param {type:"integer"}
MIN_RATING = 1200  # @param {type:"integer"}
ONLY_WINNER = True  # @param {type:"boolean"}
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
SEED = 260913  # @param {type:"integer"}
PORT = 8000  # @param {type:"integer"}

PKMN_REPOSITORY = "https://github.com/Iesyo/pkmn.git"
PKMN_REF = "main"
VGC_BENCH_REPOSITORY = "https://github.com/cameronangliss/vgc-bench.git"
VGC_BENCH_COMMIT = "d79f9532947ac114dce1dda2456a590afcd375b2"
NODE_VERSION = "24.21.0"

MODE = {
    "CENSUS": {"max_logs": 5000, "bc_epochs": 0, "rl_steps": 0, "num_envs": 2},
    "LIGHT": {"max_logs": 5000, "bc_epochs": 3, "rl_steps": 196_608, "num_envs": 2},
    "NORMAL": {"max_logs": 20_000, "bc_epochs": 5, "rl_steps": 983_040, "num_envs": 4},
    "HARD": {"max_logs": 50_000, "bc_epochs": 10, "rl_steps": 2_949_120, "num_envs": 4},
}[RUN_MODE]

if RUN_MODE == "CENSUS":
    RUN_RL = False
    RUN_BC_IF_ENOUGH_DATA = False

print("Modo:", RUN_MODE, MODE)


## 2. Drive y rutas canónicas

La libreta no usa `/content` como única copia. El corpus, replays, trayectorias, baseline, checkpoints y reportes viven en Drive.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Colabs/LikeNoOneEverWas/BattleLab/MC-Training")
DATA_ROOT = DRIVE_ROOT / "data"
TEAM_DIR = DATA_ROOT / "teams" / "vgcpastes-champions-mc"
OUTPUT_ROOT = DRIVE_ROOT / "training"
LOG_ROOT = DRIVE_ROOT / "logs"
PKMN_ROOT = Path("/content/pkmn")
VGC_BENCH_ROOT = Path("/content/vgc-bench")
RUNTIME_ROOT = Path("/content/battle-lab-runtime")
SHOWDOWN_ROOT = RUNTIME_ROOT / "pokemon-showdown"

for path in (DATA_ROOT, TEAM_DIR, OUTPUT_ROOT, LOG_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print("📁 Root persistente:", DRIVE_ROOT)


## 3. Preparar código, Python, Node y runtimes fijados

Esta celda fija las revisiones reales y registra el SHA de `pkmn` usado. VGC-Bench se deja en detached HEAD; M-C se inyecta en runtime sin editar su código fuente.


In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path

started = time.monotonic()

def run(command, *, cwd=None, label=None):
    if label:
        print(f"\n▶ {label}", flush=True)
    subprocess.run([str(x) for x in command], cwd=cwd, check=True)

# Node 24 para el Showdown actual.
node_major = int(subprocess.check_output(["node", "-p", "process.versions.node.split('.')[0]"], text=True).strip())
if node_major < 24:
    run(["npm", "install", "-g", "n"], label="Instalando selector de Node")
    run(["n", NODE_VERSION], label=f"Instalando Node {NODE_VERSION}")
    os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
print("Node:", subprocess.check_output(["node", "--version"], text=True).strip())

# Repo del proyecto.
if not (PKMN_ROOT / ".git").is_dir():
    run(["git", "clone", "--filter=blob:none", PKMN_REPOSITORY, str(PKMN_ROOT)], label="Clonando pkmn")
run(["git", "fetch", "origin", PKMN_REF], cwd=PKMN_ROOT, label="Actualizando pkmn")
run(["git", "checkout", "--force", "FETCH_HEAD"], cwd=PKMN_ROOT, label="Fijando pkmn para esta ejecución")
pkmn_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PKMN_ROOT, text=True).strip()
print("pkmn SHA:", pkmn_sha)

# Dependencias del laboratorio + entrenamiento, reutilizando el PyTorch GPU de Colab.
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PKMN_ROOT / "battle_lab" / "requirements-phase1.txt")], label="Instalando poke-env fijado")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PKMN_ROOT / "battle_lab" / "requirements-mc-train.txt")], label="Instalando dependencias de entrenamiento")

# Showdown fijado por el proyecto.
sys.path.insert(0, str(PKMN_ROOT))
from battle_lab.showdown_smoke import (
    DEFAULT_SHOWDOWN_REPOSITORY,
    ensure_showdown_checkout,
    install_runtime_config,
    read_showdown_commit,
)
showdown_commit = read_showdown_commit()
ensure_showdown_checkout(
    checkout=SHOWDOWN_ROOT,
    repository=DEFAULT_SHOWDOWN_REPOSITORY,
    commit=showdown_commit,
    logs_dir=LOG_ROOT,
)
install_runtime_config(SHOWDOWN_ROOT)
print("Showdown SHA:", showdown_commit)

# VGC-Bench exacto usado por Battle Lab.
from battle_lab.vgc_bench_battle import ensure_vgc_bench_checkout
ensure_vgc_bench_checkout(
    checkout=VGC_BENCH_ROOT,
    repository=VGC_BENCH_REPOSITORY,
    commit=VGC_BENCH_COMMIT,
)
print("VGC-Bench SHA:", VGC_BENCH_COMMIT)
print(f"✅ Preparación lista en {time.monotonic() - started:.1f}s")


## 4. Snapshot completo de VGCPastes M-C

Descarga la pestaña `Champions M-C`, obtiene cada Poképaste, deduplica por contenido, valida legalidad con el Showdown M-C fijado y guarda `manifest.json`. El snapshot inicial esperado es de 184 equipos, pero la libreta acepta que el repositorio crezca.


In [ ]:
import subprocess, sys

BASE = [
    sys.executable, "-m", "battle_lab.mc_training",
    "--vgc-bench", str(VGC_BENCH_ROOT),
    "--project-root", str(PKMN_ROOT),
    "--data-root", str(DATA_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--team-dir", str(TEAM_DIR),
    "--showdown", str(SHOWDOWN_ROOT),
    "--port", str(PORT),
    "--device", DEVICE,
    "--seed", str(SEED),
]

CENSUS_BASE = [
    sys.executable, "-m", "battle_lab.mc_census",
    "--vgc-bench", str(VGC_BENCH_ROOT),
    "--data-root", str(DATA_ROOT),
    "--team-dir", str(TEAM_DIR),
    "--output-root", str(OUTPUT_ROOT),
]

def run_pipeline(base, *args):
    command = base + [str(x) for x in args]
    print("\n$", " ".join(command), "\n", flush=True)
    subprocess.run(command, check=True, cwd=PKMN_ROOT)

def mc(*args):
    run_pipeline(BASE, *args)

def census(*args):
    run_pipeline(CENSUS_BASE, *args)

if SYNC_TEAMS:
    mc("sync-teams", "--minimum-teams", 150)
else:
    print("⏭️ Reutilizando snapshot de equipos existente")


## 5. Replays humanos M-C y trayectorias BC

Se usa **el scraper oficial de VGC-Bench** con el formato M-C y M-C Bo3. Conserva solamente replays con seis Pokémon por lado, Open Team Sheet, Turn 1 válido y equipos distinguibles; son los mismos criterios estructurales que M-A/M-B.

Luego `logs2trajs.py` reconstruye estados/acciones. Por defecto filtramos perspectivas con rating ≥1200 y solo ganadores para evitar enseñar demasiado juego basura. Si la regulación todavía es demasiado nueva y no alcanza el umbral, BC-MC se omite automáticamente.


In [ ]:
if SCRAPE_HUMAN_LOGS:
    census(
        "scrape",
        "--max-logs-per-format", MODE["max_logs"],
        "--num-workers", 16,
        "--read-increment", min(20_000, MODE["max_logs"]),
    )
else:
    print("⏭️ Reutilizando battle logs existentes")

logs_manifest_path = DATA_ROOT / "logs_manifest.json"
logs_manifest = json.loads(logs_manifest_path.read_text()) if logs_manifest_path.exists() else {}
accepted_logs = int(logs_manifest.get("totalLogs", 0))
print(f"Replays OTS M-C aceptados: {accepted_logs:,}")

if BUILD_TRAJECTORIES and accepted_logs > 0:
    args = ["build-trajectories", "--num-workers", max(2, (os.cpu_count() or 2) // 2)]
    if MIN_RATING > 0:
        args += ["--min-rating", MIN_RATING]
    if ONLY_WINNER:
        args += ["--only-winner"]
    mc(*args)
elif BUILD_TRAJECTORIES:
    print("⚠️ No hay logs M-C aceptados todavía; no se pueden construir trayectorias.")
else:
    print("⏭️ Reutilizando trayectorias existentes")

census("report")


## 6. Decidir BC-MC y entrenar PPO/self-play

Si existen suficientes demostraciones humanas, primero hacemos **Behavior Cloning adicional sobre el checkpoint público**, nunca desde cero. Luego PPO/self-play continúa sobre **todos los equipos M-C válidos del snapshot de VGCPastes**.

Si aún no hay suficientes replays humanos, se salta BC-MC y PPO parte directamente del checkpoint público M-A/M-B. Así no bloqueamos el entrenamiento por lo joven de la regulación, pero tampoco fingimos que el dataset público ya tiene M-C.


In [ ]:
import json
from battle_lab.showdown_smoke import running_showdown

traj_manifest_path = DATA_ROOT / "trajs_manifest.json"
traj_manifest = json.loads(traj_manifest_path.read_text()) if traj_manifest_path.exists() else {}
traj_count = int(traj_manifest.get("trajectories", 0))
transition_count = int(traj_manifest.get("transitions", 0))
has_bc_data = traj_count >= MIN_BC_TRAJECTORIES and transition_count >= MIN_BC_TRANSITIONS
print(f"Trayectorias M-C: {traj_count:,} · transiciones: {transition_count:,} · BC habilitable: {has_bc_data}")

bc_checkpoint = None
need_server = (RUN_BC_IF_ENOUGH_DATA and has_bc_data and MODE["bc_epochs"] > 0) or (RUN_RL and MODE["rl_steps"] > 0)

if need_server:
    with running_showdown(SHOWDOWN_ROOT, PORT, LOG_ROOT / "mc-training-showdown.log"):
        if RUN_BC_IF_ENOUGH_DATA and has_bc_data and MODE["bc_epochs"] > 0:
            mc(
                "bc",
                "--epochs", MODE["bc_epochs"],
                "--div-frac", 0.1,
                "--eval-battles", 50,
            )
            bc_summary = json.loads((OUTPUT_ROOT / "bc" / "summary.json").read_text())
            bc_checkpoint = Path(bc_summary["finalCheckpoint"])
            print("🧠 BC-MC listo:", bc_checkpoint)
        elif RUN_BC_IF_ENOUGH_DATA and not has_bc_data:
            print("⚠️ Aún no hay suficientes demostraciones humanas M-C; se conserva el baseline y se pasa a self-play.")

        if RUN_RL and MODE["rl_steps"] > 0:
            rl_args = [
                "rl",
                "--total-steps", MODE["rl_steps"],
                "--num-envs", MODE["num_envs"],
                "--num-eval-workers", 4,
            ]
            if bc_checkpoint is not None:
                rl_args += ["--initial-checkpoint", str(bc_checkpoint)]
            mc(*rl_args)
else:
    print("ℹ️ Modo censo: no se inicia el servidor ni entrenamiento.")


## 7. Resumen final persistente

Esta celda deja el estado de la corrida listo para revisión: número de equipos, logs, trayectorias, baseline/BC usado y último checkpoint RL. El benchmark final contra el baseline original se ejecutará con el **Battle Lab** de evaluación, no dentro de este entrenamiento, para no mezclar entrenamiento y examen.


In [ ]:
import json
from pathlib import Path

census("report")
summary = json.loads((OUTPUT_ROOT / "census.json").read_text())
rl_summary_path = OUTPUT_ROOT / "rl" / "summary.json"
bc_summary_path = OUTPUT_ROOT / "bc" / "summary.json"

print("\n===== BATTLE LAB M-C =====")
print("Equipos VGCPastes válidos:", summary["teams"]["usable"])
print("Replays candidatos revisados:", summary["candidateReplays"])
print("Logs humanos M-C aceptados:", summary["humanLogs"])
print("Descartes por causa:", summary["discardByCause"])
print("Trayectorias:", summary["trajectories"])
print("Transiciones:", summary["transitions"])
print("Elo:", summary["ratings"])
print("Cobertura de especies:", summary["speciesCoverage"]["sourceCoverageRate"])
print("Firmas de equipo observadas:", summary["teamCoverage"]["observedUniqueSignatures"])
print("Cores/arquetipos proxy de 3 especies:", summary["archetypeCoreCoverage"]["topCores3"][:10])
print("Espacio actual (bytes):", summary["storage"]["dataBytes"] + summary["storage"]["teamSnapshotBytes"])
if bc_summary_path.exists():
    bc_summary = json.loads(bc_summary_path.read_text())
    print("BC-MC:", bc_summary["finalCheckpoint"])
if rl_summary_path.exists():
    rl_summary = json.loads(rl_summary_path.read_text())
    print("RL M-C final:", rl_summary["finalCheckpoint"])
    print("SHA256:", rl_summary["finalCheckpointSha256"])
print("Drive:", DRIVE_ROOT)
if RUN_MODE == "CENSUS":
    print("\n✅ Censo listo. Revisa estos números antes de cambiar RUN_MODE a LIGHT.")
else:
    print("\n✅ Siguiente paso: evaluar el último checkpoint M-C contra el baseline original en el Battle Lab calibrado.")
